In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms


# 加载数据并处理为tensor

In [13]:
# 加载CIFAR-10数据集
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

# 定义CIFAR-10数据集类
class CIFAR10Dataset(Dataset):
    def __init__(self, img_dir, labels_file, transform=None):
        self.img_dir = img_dir
        self.transform = transform

        # 读取标签文件，read_csv默认读取第一行作为列名
        self.labels_df = pd.read_csv(labels_file)
        self.img_names = self.labels_df.iloc[:, 0].values.astype(str)  # 第一列是图片名称，确保为字符串类型

        # 类别名称字典，使用字典可以提高查找速度
        self.class_names_dict = {'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3,
                                 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}
        # 将文本标签转换为数字ID
        self.labels = [self.class_names_dict[label] for label in self.labels_df.iloc[:, 1].values]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx] + '.png') #图片路径
        image = Image.open(img_path) #打开图片
        label = self.labels[idx]

        if self.transform:
            image_tensor = self.transform(image)

        return image_tensor, label

# 定义数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4917, 0.4823, 0.4467), (0.2024, 0.1995, 0.2010))
])

In [14]:

# 加载CIFAR-10数据集
# img_dir = r"competitions/cifar-10/train"
# labels_file = r"./trainLabels.csv"
img_dir = r"D:\cifar-10\train\train"
labels_file = r"D:\cifar-10\trainLabels.csv"
full_dataset = CIFAR10Dataset(img_dir=img_dir, labels_file=labels_file, transform=transform)

# 定义类别名称
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# 划分训练集和验证集
train_size = 45000
val_size = 5000
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

# 查看数据集基本信息
print(f"完整数据集大小: {len(full_dataset)}")
print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")


完整数据集大小: 50000
训练集大小: 45000
验证集大小: 5000


# 把数据集划分为训练集45000和验证集5000，并给DataLoader

In [15]:

# 创建数据加载器
batch_size = 64
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True #打乱数据集，每次迭代时，数据集的顺序都会被打乱
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

# 搭建模型

In [16]:

class BasicBlock(nn.Module):
    """
    ResNet18的基本残差块
    """
    expansion = 1  # 残差块输出通道扩展倍数

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        """
        参数:
            in_channels: 输入通道数
            out_channels: 输出通道数
            stride: 步幅
            downsample: 下采样层（用于匹配维度）
        """
        super().__init__()
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # 如果输入和输出通道数不同或步长不为1，则需要使用1x1卷积进行调整
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

       #下采样层指示是否需要下采样，即卷积核大小为3x3，步长为2，填充为1
       # 与self.shortcut类似,显示结构的下采样层逻辑，这种设计允许更灵活地定制残差连接，特别是在构建复杂网络架构（如 ResNet 变体）时
        self.downsample = downsample  # 下采样层（如果需要调整维度），自定义在卷积块中进行下采样
       #downsample = nn.Sequential(nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False),nn.BatchNorm2d(128))

    def forward(self, x):
        """
        前向传播
        """
        # identity即residual
        identity = x  # 保存输入用于残差连接

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)  # 第一次激活

        out = self.conv2(out)
        out = self.bn2(out)
        # 如果需要下采样，则对identity进行下采样
        #if self.downsample is not None:identity = self.downsample(x)
        out += self.shortcut(identity)  # 残差连接
        out = self.relu(out)  # 第二次激活

        return out



In [17]:
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    """
    ResNet18的基本残差块
    """
    expansion = 1  # 残差块输出通道扩展倍数

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        """
        参数:
            in_channels: 输入通道数
            out_channels: 输出通道数
            stride: 步幅
            downsample: 下采样层（用于匹配维度）
        """
        super().__init__()
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # 如果输入和输出通道数不同或步长不为1，则需要使用1x1卷积进行调整
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

       #下采样层指示是否需要下采样，即卷积核大小为3x3，步长为2，填充为1
       # 与self.shortcut类似,显示结构的下采样层逻辑，这种设计允许更灵活地定制残差连接，特别是在构建复杂网络架构（如 ResNet 变体）时
        self.downsample = downsample  # 下采样层（如果需要调整维度），自定义在卷积块中进行下采样
       #downsample = nn.Sequential(nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False),nn.BatchNorm2d(128))

    def forward(self, x):
        """
        前向传播
        """
        # identity即residual
        identity = x  # 保存输入用于残差连接

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)  # 第一次激活

        out = self.conv2(out)
        out = self.bn2(out)
        # 如果需要下采样，则对identity进行下采样
        #if self.downsample is not None:identity = self.downsample(x)
        out += self.shortcut(identity)  # 残差连接
        out = self.relu(out)  # 第二次激活

        return out


class Resnet18Network(nn.Module):
    def __init__(self, activation='relu',num_classes=10):
        """
        参数:
            activation: 激活函数类型，默认为'relu'，可选'selu'
        """
        super().__init__()#

        # 选择激活函数
        if activation.lower() == 'selu':
            act_layer = nn.SELU  # SELU激活
        else:
            act_layer = nn.ReLU  # 默认ReLU激活

        # 第一组卷积层 - 使用Sequential组织
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),  # 输入通道3，输出通道128，卷积核3x3，padding=1
            nn.BatchNorm2d(64),
            act_layer(),
            nn.MaxPool2d(kernel_size=2, stride=2)         # 最大池化，尺寸减半
        )
        # 定义残差层和全局平均池化和全连接层
        # 第二组卷积块：包含2个残差块，输入通道64，输出通道128
        self.conv_block2 = nn.Sequential(
            BasicBlock(64, 64, stride=2),  # 第一个残差块，下采样
            BasicBlock(64, 64)            # 第二个残差块
        )
        # 第三组卷积块：包含2个残差块，输入通道128，输出通道256
        self.conv_block3 = nn.Sequential(
            BasicBlock(64, 128, stride=2),  # 第一个残差块，下采样
            BasicBlock(128, 128)             # 第二个残差块
        )

        # 第四组卷积块：包含2个残差块，输入通道256，输出通道512
        self.conv_block4 = nn.Sequential(
            BasicBlock(128, 256, stride=2),  # 第一个残差块，下采样
            BasicBlock(256, 256)             # 第2个残差块
        )

        # 第五组卷积块：包含2个残差块，输入通道256，输出通道512
        self.conv_block5 = nn.Sequential(
            BasicBlock(256, 512, stride=2),  # 第一个残差块，下采样
            BasicBlock(512,512)             # 第2个残差块
        )


        # 全局平均池化层
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))  # 输出为(batch, 通道, 1, 1)

        # 全连接分类器
        self.classifier = nn.Sequential(
            nn.Linear(512, num_classes)  # 512为最后通道数，num_classes默认10为CIFAR-10类别数
        )

        # 中文注释：定义了两组残差块用于特征提取，使用全局平均池化降维，最后通过全连接层输出分类结果




    def forward(self, x):
        # 前向传播使用Sequential定义的块
        x = self.conv_block1(x)  # 第一组卷积块
        x = self.conv_block2(x)  # 第二组卷积块
        x = self.conv_block3(x)  # 第三组卷积块
        x = self.conv_block4(x)  # 第四组卷积块
        x = self.conv_block5(x)  # 第五组卷积块


        # 全局平均池化
        x = self.global_avg_pool(x)
        x = torch.flatten(x,1)              # 展平
        # 分类器
        x = self.classifier(x)  # 全连接分类器

        return x  # 输出结果

In [18]:
# 实例化模型
model = Resnet18Network()

# 从train_loader获取第一个批次的数据
dataiter = iter(train_loader)
images, labels = next(dataiter)

# 查看批次数据的形状
print("批次图像形状:", images.shape)
print("批次标签形状:", labels.shape)


print('-'*100)
# 进行前向传播
with torch.no_grad():  # 不需要计算梯度
    outputs = model(images)


print(outputs.shape)


批次图像形状: torch.Size([64, 3, 32, 32])
批次标签形状: torch.Size([64])
----------------------------------------------------------------------------------------------------
torch.Size([64, 10])


In [19]:
# 计算模型的总参数量
# 统计需要求梯度的参数总量
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"需要求梯度的参数总量: {total_params}")

# 统计所有参数总量
all_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {all_params}")

# 查看每层参数量明细
print("\n各层参数量明细:")
for name, param in model.named_parameters():
    print(f"{name}: {param.numel()} 参数")

需要求梯度的参数总量: 11178250
模型总参数量: 11178250

各层参数量明细:
conv_block1.0.weight: 1728 参数
conv_block1.0.bias: 64 参数
conv_block1.1.weight: 64 参数
conv_block1.1.bias: 64 参数
conv_block2.0.conv1.weight: 36864 参数
conv_block2.0.bn1.weight: 64 参数
conv_block2.0.bn1.bias: 64 参数
conv_block2.0.conv2.weight: 36864 参数
conv_block2.0.bn2.weight: 64 参数
conv_block2.0.bn2.bias: 64 参数
conv_block2.0.shortcut.0.weight: 4096 参数
conv_block2.0.shortcut.1.weight: 64 参数
conv_block2.0.shortcut.1.bias: 64 参数
conv_block2.1.conv1.weight: 36864 参数
conv_block2.1.bn1.weight: 64 参数
conv_block2.1.bn1.bias: 64 参数
conv_block2.1.conv2.weight: 36864 参数
conv_block2.1.bn2.weight: 64 参数
conv_block2.1.bn2.bias: 64 参数
conv_block3.0.conv1.weight: 73728 参数
conv_block3.0.bn1.weight: 128 参数
conv_block3.0.bn1.bias: 128 参数
conv_block3.0.conv2.weight: 147456 参数
conv_block3.0.bn2.weight: 128 参数
conv_block3.0.bn2.bias: 128 参数
conv_block3.0.shortcut.0.weight: 8192 参数
conv_block3.0.shortcut.1.weight: 128 参数
conv_block3.0.shortcut.1.bias: 128 参数
conv_bl